# 选择 Chunk Size（第二节）

目标：在相同数据与评测问题集下，对比不同 `chunk_size`/`chunk_overlap` 的检索与回答质量，并得到一组可复用的默认参数。

## 0) 导入依赖并加载环境变量

In [4]:
from __future__ import annotations

import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv("../.env")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


## 1) 读取文档

In [5]:
path = Path("../data/Understanding_Climate_Change.pdf")
assert path.exists(), f"找不到 PDF: {path.resolve()}"

docs = PyPDFLoader(str(path)).load()
len(docs)

33

## 2) 构造评测问题集

In [9]:
llm_q = ChatOpenAI(model="gpt-5.4-mini", temperature=0)

prompt_q = ChatPromptTemplate.from_messages(
    [
        ("system", "Generate diverse evaluation questions based on the provided document excerpt. Return one question per line."),
        ("human", "Generate {n} questions.\n\nDocument excerpt:\n{excerpt}"),
    ]
)

excerpt = "\n\n".join(d.page_content for d in docs[:20])

n_questions = 10

questions_text = (prompt_q | llm_q).invoke({"n": n_questions, "excerpt": excerpt}).content
eval_questions = [q.strip(" -\t") for q in questions_text.splitlines() if q.strip()]
eval_questions[:5], len(eval_questions)

(['1. What are the main human activities identified as contributors to climate change in the excerpt?',
  '2. How do greenhouse gases like carbon dioxide, methane, and nitrous oxide contribute to the greenhouse effect?',
  '3. Why is coal described as the most carbon-intensive fossil fuel, and what does this imply for emissions?',
  '4. What evidence do scientists use to reconstruct past climate conditions and predict future trends?',
  '5. How does deforestation worsen climate change, and why are tropical rainforests especially important?'],
 10)

## 3) 定义评估函数（对每个 chunk size）

In [10]:
llm_answer = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
llm_judge = ChatOpenAI(model="gpt-5.4-mini", temperature=0)

prompt_answer = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer the question using only the provided context. If not in context, say you don't know."),
        ("human", "Context:\n{context}\n\nQuestion: {question}"),
    ]
)

prompt_judge = ChatPromptTemplate.from_messages(
    [
        ("system", "You evaluate retrieval quality. Output JSON with keys Relevance, Completeness, Conciseness (1-5)."),
        ("human", "Question: {question}\n\nRetrieved Context:\n{context}"),
    ]
)


def evaluate_chunking(chunk_size: int, chunk_overlap: int, k: int, questions: list[str]):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True,
    )
    splits = splitter.split_documents(docs)

    vectorstore = FAISS.from_documents(splits, OpenAIEmbeddings())
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})

    judge_results: list[str] = []
    total_time = 0.0

    for q in questions:
        t0 = time.time()
        retrieved_docs = retriever.invoke(q)
        context = "\n\n".join(d.page_content for d in retrieved_docs)
        _ = (prompt_answer | llm_answer).invoke({"context": context, "question": q}).content
        judge = (prompt_judge | llm_judge).invoke({"context": context, "question": q}).content
        total_time += time.time() - t0
        judge_results.append(judge)

    avg_time = total_time / max(1, len(questions))
    return {
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "k": k,
        "num_splits": len(splits),
        "avg_time_sec": avg_time,
        "judge_results": judge_results,
    }

## 4) 测试不同 chunk size

In [11]:
chunk_sizes = [128, 256]
k = 2

results = []
for cs in chunk_sizes:
    r = evaluate_chunking(chunk_size=cs, chunk_overlap=cs // 5, k=k, questions=eval_questions)
    results.append(r)
    print(
        f"chunk_size={r['chunk_size']} overlap={r['chunk_overlap']} splits={r['num_splits']} avg_time={r['avg_time_sec']:.2f}s"
    )

results

chunk_size=128 overlap=25 splits=795 avg_time=9.43s
chunk_size=256 overlap=51 splits=364 avg_time=11.03s


[{'chunk_size': 128,
  'chunk_overlap': 25,
  'k': 2,
  'num_splits': 795,
  'avg_time_sec': 9.433307147026062,
  'judge_results': ['{"Relevance":5,"Completeness":4,"Conciseness":5}',
   '{"Relevance":2,"Completeness":1,"Conciseness":5}',
   '{"Relevance":5,"Completeness":2,"Conciseness":4}',
   '{"Relevance":3,"Completeness":2,"Conciseness":5}',
   '{"Relevance":2,"Completeness":1,"Conciseness":5}',
   '{"Relevance":2,"Completeness":1,"Conciseness":5}',
   '{"Relevance":4,"Completeness":2,"Conciseness":4}',
   '{"Relevance":2,"Completeness":1,"Conciseness":4}',
   '{"Relevance":1,"Completeness":1,"Conciseness":5}',
   '{"Relevance":2,"Completeness":1,"Conciseness":5}']},
 {'chunk_size': 256,
  'chunk_overlap': 51,
  'k': 2,
  'num_splits': 364,
  'avg_time_sec': 11.028538489341736,
  'judge_results': ['{"Relevance":5,"Completeness":4,"Conciseness":5}',
   '{"Relevance":5,"Completeness":3,"Conciseness":5}',
   '{"Relevance":5,"Completeness":4,"Conciseness":5}',
   '{"Relevance":4,"Comp